
## Folder 05 — File 04: Business Rule Validation:


We will validate:

- Order total against order-item totals.
- Payment amounts against order totals.
- Multiple payment records for an order.
- Shipment records against orders.
- Delivered shipment date consistency.
- Inventory quantities and snapshots.
- Product prices against product costs.
- Orders with missing order items.


**Important:**
 These are business rules, not universal truths. For example, an order can legitimately have multiple partial payments. We should not automatically flag every multiple-payment order as invalid.

In [0]:

# Databricks notebook source

# MAGIC %md
# MAGIC # 08.2 - Business Rule Remediation and Validation
# MAGIC
# MAGIC Objectives:
# MAGIC 1. Identify invalid business relationships across Silver tables.
# MAGIC 2. Quarantine invalid parent records before child records.
# MAGIC 3. Preserve original records before deleting them from Silver.
# MAGIC 4. Verify that all targeted business rules pass after remediation.
# MAGIC 5. Save a fresh remediation report with a unique run ID.
# MAGIC
# MAGIC IMPORTANT:
# MAGIC - This notebook physically deletes quarantined records from Silver.
# MAGIC - Existing quarantine records are preserved.
# MAGIC - Existing quarantine tables are not overwritten.
# MAGIC - Pending or unpaid orders are not automatically treated as invalid.
# MAGIC - A PASS is reported only when the corresponding check returns zero.

# COMMAND ----------

# 1. IMPORT LIBRARIES

from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timezone
import re

# COMMAND ----------

# 2. CONFIGURATION

CATALOG = "workspace"
SILVER_SCHEMA = "ecommerce_silver"

SILVER = f"{CATALOG}.{SILVER_SCHEMA}"

TABLES = {
    "customers": f"{SILVER}.silver_customers",
    "products": f"{SILVER}.silver_products",
    "orders": f"{SILVER}.silver_orders",
    "order_items": f"{SILVER}.silver_order_items",
    "payments": f"{SILVER}.silver_payments",
    "shipments": f"{SILVER}.silver_shipments",
    "inventory": f"{SILVER}.silver_inventory"
}

QUARANTINE_TABLES = {
    "customers": f"{SILVER}.quarantine_customers",
    "products": f"{SILVER}.quarantine_products",
    "orders": f"{SILVER}.quarantine_orders",
    "order_items": f"{SILVER}.quarantine_order_items",
    "payments": f"{SILVER}.quarantine_payments",
    "shipments": f"{SILVER}.quarantine_shipments",
    "inventory": f"{SILVER}.quarantine_inventory"
}

BUSINESS_KEYS = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "payments": "payment_id",
    "shipments": "shipment_id",
    "inventory": "inventory_snapshot_id"
}

REMEDIATION_REPORT_TABLE = (
    f"{SILVER}.business_rule_remediation_results"
)

TOLERANCE = 0.02

RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat()

RUN_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

# COMMAND ----------

# 3. REQUIRED SCHEMAS

REQUIRED_COLUMNS = {
    "customers": [
        "customer_id"
    ],
    "products": [
        "product_id"
    ],
    "orders": [
        "order_id",
        "customer_id",
        "order_status",
        "order_total"
    ],
    "order_items": [
        "order_item_id",
        "order_id",
        "product_id",
        "line_total"
    ],
    "payments": [
        "payment_id",
        "order_id"
    ],
    "shipments": [
        "shipment_id",
        "order_id"
    ],
    "inventory": [
        "inventory_snapshot_id",
        "product_id"
    ]
}


# COMMAND ----------

# 4. LOAD SILVER TABLES

def load_silver_tables():

    loaded = {}

    for dataset, table_name in TABLES.items():

        if not spark.catalog.tableExists(table_name):

            raise Exception(
                f"Silver table does not exist: {table_name}"
            )

        df = spark.table(table_name)

        missing_columns = [
            column
            for column in REQUIRED_COLUMNS[dataset]
            if column not in df.columns
        ]

        if missing_columns:

            raise Exception(
                f"{table_name} is missing columns: "
                f"{missing_columns}"
            )

        loaded[dataset] = df

    return loaded


silver_dfs = load_silver_tables()


# COMMAND ----------

# 5. INITIAL SILVER COUNT

print("=" * 80)
print("INITIAL SILVER TABLE VERIFICATION")
print("=" * 80)

before_counts = {}

for dataset, df in silver_dfs.items():

    count = df.count()

    before_counts[dataset] = count

    print(
        f"{dataset:15} : {count:,} records"
    )


# COMMAND ----------

# 6. VERIFY QUARANTINE TABLES

for dataset, quarantine_table in QUARANTINE_TABLES.items():

    if not spark.catalog.tableExists(quarantine_table):

        raise Exception(
            f"Missing quarantine table: {quarantine_table}"
        )

    business_key = BUSINESS_KEYS[dataset]

    quarantine_df = spark.table(quarantine_table)

    if business_key not in quarantine_df.columns:

        raise Exception(
            f"{quarantine_table} is missing "
            f"business key {business_key}"
        )

    print(
        f"Verified quarantine table: {quarantine_table}"
    )


# COMMAND ----------

# 7. GENERIC ORPHAN DETECTION

def find_orphans(
    child_df,
    parent_df,
    child_key,
    parent_key
):

    parent_keys = (
        parent_df
        .select(
            F.col(parent_key).alias("_parent_key")
        )
        .filter(
            F.col("_parent_key").isNotNull()
        )
        .distinct()
    )

    orphan_df = (
        child_df.alias("child")
        .join(
            parent_keys.alias("parent"),
            F.col(f"child.{child_key}")
            == F.col("parent._parent_key"),
            "left_anti"
        )
    )

    return orphan_df


# COMMAND ----------

# 8. COMBINE MULTIPLE QUARANTINE REASONS

def combine_issues(
    source_df,
    business_key,
    issue_frames
):

    combined = None

    for issue_df, reason in issue_frames:

        current = (
            issue_df
            .select(business_key)
            .filter(
                F.col(business_key).isNotNull()
            )
            .distinct()
            .withColumn(
                "_reason",
                F.lit(reason)
            )
        )

        combined = (
            current
            if combined is None
            else combined.unionByName(current)
        )

    if combined is None:

        return (
            source_df.limit(0)
            .withColumn(
                "quarantine_reason",
                F.lit(None).cast("string")
            )
        )

    aggregated = (
        combined
        .groupBy(business_key)
        .agg(
            F.concat_ws(
                "; ",
                F.sort_array(
                    F.collect_set("_reason")
                )
            ).alias("quarantine_reason")
        )
    )

    result = (
        source_df.alias("source")
        .join(
            aggregated.alias("issues"),
            F.col(f"source.{business_key}")
            == F.col(f"issues.{business_key}"),
            "inner"
        )
        .select(
            "source.*",
            F.col("issues.quarantine_reason")
        )
    )

    return result


# COMMAND ----------

# 9. PREPARE QUARANTINE SCHEMA

def ensure_quarantine_reason_column(table_name):

    schema = spark.table(table_name).schema

    columns = {
        field.name.lower()
        for field in schema
    }

    if (
        "quarantine_reason" not in columns
        and "reason" not in columns
    ):

        spark.sql(
            f"""
            ALTER TABLE {table_name}
            ADD COLUMNS (
                quarantine_reason STRING
            )
            """
        )

    return spark.table(table_name).schema


def prepare_quarantine(
    source_df,
    quarantine_table
):

    target_schema = ensure_quarantine_reason_column(
        quarantine_table
    )

    target_columns = {
        field.name
        for field in target_schema
    }

    output = source_df

    metadata_values = {

        "quarantine_reason":
            F.col("quarantine_reason"),

        "reason":
            F.col("quarantine_reason"),

        "status":
            F.lit("QUARANTINED"),

        "validation_status":
            F.lit("FAIL"),

        "quarantined_at":
            F.current_timestamp(),

        "_quarantined_at":
            F.current_timestamp(),

        "remediation_run_id":
            F.lit(RUN_ID),

        "run_id":
            F.lit(RUN_ID)
    }

    for column_name, expression in metadata_values.items():

        if (
            column_name in target_columns
            and column_name not in output.columns
        ):

            output = output.withColumn(
                column_name,
                expression
            )

    original_columns = {
        column
        for column in source_df.columns
        if column != "quarantine_reason"
    }

    missing_original = (
        original_columns - target_columns
    )

    if missing_original:

        raise Exception(
            f"{quarantine_table} cannot preserve original "
            f"columns: {sorted(missing_original)}"
        )

    select_expressions = []

    for field in target_schema:

        if field.name in output.columns:

            select_expressions.append(
                F.col(field.name)
                .cast(field.dataType)
                .alias(field.name)
            )

        else:

            if not field.nullable:

                raise Exception(
                    f"Cannot populate mandatory column "
                    f"{field.name} in {quarantine_table}"
                )

            select_expressions.append(
                F.lit(None)
                .cast(field.dataType)
                .alias(field.name)
            )

    return output.select(*select_expressions)


# COMMAND ----------

# 10. SAFE QUARANTINE AND SILVER DELETE

def quarantine_records(
    candidates,
    quarantine_table,
    silver_table,
    business_key,
    label
):

    if business_key not in candidates.columns:

        raise Exception(
            f"{label}: missing business key {business_key}"
        )

    candidate_count = candidates.count()

    if candidate_count == 0:

        print(
            f"No records to quarantine: {label}"
        )

        return 0

    # Null keys cannot be safely deleted or reconciled.

    null_keys = (
        candidates
        .filter(
            F.col(business_key).isNull()
        )
        .count()
    )

    if null_keys > 0:

        raise Exception(
            f"{label}: {null_keys} null business keys found. "
            "Deletion cancelled."
        )

    # Duplicate business keys must be investigated.
    # Do not silently drop them.

    duplicate_keys = (
        candidates
        .groupBy(business_key)
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    if duplicate_keys > 0:

        raise Exception(
            f"{label}: {duplicate_keys} duplicate candidate keys. "
            "Deletion cancelled."
        )

    # Add fallback reason if the caller did not provide one.

    if "quarantine_reason" not in candidates.columns:

        candidates = candidates.withColumn(
            "quarantine_reason",
            F.lit(label)
        )

    # Identify keys already preserved in quarantine.

    existing_keys = (
        spark.table(quarantine_table)
        .select(business_key)
        .filter(
            F.col(business_key).isNotNull()
        )
        .distinct()
    )

    new_records = (
        candidates
        .join(
            existing_keys,
            business_key,
            "left_anti"
        )
    )

    new_count = new_records.count()

    # Append only new quarantine records.

    if new_count > 0:

        prepared = prepare_quarantine(
            new_records,
            quarantine_table
        )

        (
            prepared.write
            .format("delta")
            .mode("append")
            .saveAsTable(quarantine_table)
        )

    # Verify every candidate key exists in quarantine.

    persisted_keys = (
        spark.table(quarantine_table)
        .select(business_key)
        .filter(
            F.col(business_key).isNotNull()
        )
        .distinct()
    )

    missing_persisted = (
        candidates
        .select(business_key)
        .distinct()
        .join(
            persisted_keys,
            business_key,
            "left_anti"
        )
    )

    missing_count = missing_persisted.count()

    if missing_count > 0:

        raise Exception(
            f"{label}: {missing_count} records were not "
            "preserved in quarantine. Silver deletion cancelled."
        )

    # Delete only verified business keys.

    view_name = (
        "remediation_keys_"
        + re.sub(
            r"[^A-Za-z0-9_]",
            "_",
            business_key
        )
    )

    (
        candidates
        .select(business_key)
        .distinct()
        .createOrReplaceTempView(view_name)
    )

    spark.sql(
        f"""
        DELETE FROM {silver_table}
        WHERE {business_key} IN (
            SELECT {business_key}
            FROM {view_name}
        )
        """
    )

    # Verify deletion.

    remaining = (
        spark.table(silver_table)
        .join(
            candidates.select(business_key).distinct(),
            business_key,
            "inner"
        )
        .count()
    )

    if remaining > 0:

        raise Exception(
            f"{label}: {remaining} records remain in Silver."
        )

    print(
        f"{label}: "
        f"Candidates={candidate_count:,}, "
        f"Newly quarantined={new_count:,}, "
        f"Verified and removed={candidate_count:,}"
    )

    return candidate_count


# COMMAND ----------

# 11. IDENTIFY INVALID ORDERS
#
# Rules:
# 1. Order must reference an existing customer.
# 2. Non-cancelled order must have order items.
# 3. Order total must reconcile with item totals.
#
# Cancelled orders are allowed to have no items.

customers = silver_dfs["customers"]
products = silver_dfs["products"]
orders = silver_dfs["orders"]
order_items = silver_dfs["order_items"]


# 11.1 Orders without customers

orders_without_customers = find_orphans(
    orders,
    customers,
    "customer_id",
    "customer_id"
)


# 11.2 Non-cancelled orders without items

orders_without_items = (
    orders.alias("o")
    .join(
        order_items
        .select("order_id")
        .distinct()
        .alias("i"),
        F.col("o.order_id")
        == F.col("i.order_id"),
        "left_anti"
    )
    .filter(
        F.upper(
            F.trim(
                F.coalesce(
                    F.col("o.order_status"),
                    F.lit("")
                )
            )
        ) != "CANCELLED"
    )
)


# 11.3 Order total reconciliation

item_totals = (
    order_items
    .groupBy("order_id")
    .agg(
        F.sum("line_total").alias(
            "calculated_item_total"
        )
    )
)

order_total_mismatches = (
    orders.alias("o")
    .join(
        item_totals.alias("i"),
        F.col("o.order_id")
        == F.col("i.order_id"),
        "inner"
    )
    .filter(
        F.col("o.order_total").isNotNull()
        & F.col("i.calculated_item_total").isNotNull()
        & (
            F.abs(
                F.round(
                    F.col("o.order_total")
                    - F.col("i.calculated_item_total"),
                    2
                )
            ) > F.lit(TOLERANCE)
        )
    )
    .select("o.*")
)


# COMMAND ----------

# 12. COMBINE INVALID ORDER REASONS

orders_to_quarantine = combine_issues(
    orders,
    "order_id",
    [
        (
            orders_without_customers,
            "MISSING_CUSTOMER"
        ),
        (
            orders_without_items,
            "ORDER_WITHOUT_ITEMS"
        ),
        (
            order_total_mismatches,
            "ORDER_TOTAL_MISMATCH"
        )
    ]
)

print("=" * 80)
print("INVALID ORDER COUNTS")
print("=" * 80)

print(
    "Orders without customers:",
    orders_without_customers.count()
)

print(
    "Orders without items:",
    orders_without_items.count()
)

print(
    "Order total mismatches:",
    order_total_mismatches.count()
)

print(
    "Unique orders to quarantine:",
    orders_to_quarantine.count()
)

display(
    orders_to_quarantine.select(
        "order_id",
        "customer_id",
        "order_status",
        "order_total",
        "quarantine_reason"
    )
)


# COMMAND ----------

# 13. QUARANTINE INVALID ORDERS FIRST

quarantine_records(
    orders_to_quarantine,
    QUARANTINE_TABLES["orders"],
    TABLES["orders"],
    "order_id",
    "INVALID ORDERS"
)


# COMMAND ----------

# 14. RELOAD SILVER TABLES AFTER ORDER REMEDIATION

silver_dfs = load_silver_tables()

customers = silver_dfs["customers"]
products = silver_dfs["products"]
orders = silver_dfs["orders"]
order_items = silver_dfs["order_items"]
payments = silver_dfs["payments"]
shipments = silver_dfs["shipments"]
inventory = silver_dfs["inventory"]


# COMMAND ----------

# 15. IDENTIFY ORDER ITEMS WITHOUT ORDERS OR PRODUCTS

items_without_orders = find_orphans(
    order_items,
    orders,
    "order_id",
    "order_id"
)

items_without_products = find_orphans(
    order_items,
    products,
    "product_id",
    "product_id"
)

items_to_quarantine = combine_issues(
    order_items,
    "order_item_id",
    [
        (
            items_without_orders,
            "MISSING_ORDER"
        ),
        (
            items_without_products,
            "MISSING_PRODUCT"
        )
    ]
)

print(
    "Order items without orders:",
    items_without_orders.count()
)

print(
    "Order items without products:",
    items_without_products.count()
)

print(
    "Unique order items to quarantine:",
    items_to_quarantine.count()
)


# COMMAND ----------

# 16. QUARANTINE INVALID ORDER ITEMS

quarantine_records(
    items_to_quarantine,
    QUARANTINE_TABLES["order_items"],
    TABLES["order_items"],
    "order_item_id",
    "INVALID ORDER ITEMS"
)


# COMMAND ----------

# 17. IDENTIFY AND QUARANTINE PAYMENTS WITHOUT ORDERS

silver_dfs = load_silver_tables()

payments = silver_dfs["payments"]
orders = silver_dfs["orders"]

payments_without_orders = find_orphans(
    payments,
    orders,
    "order_id",
    "order_id"
)

print(
    "Payments without orders:",
    payments_without_orders.count()
)

quarantine_records(
    payments_without_orders,
    QUARANTINE_TABLES["payments"],
    TABLES["payments"],
    "payment_id",
    "PAYMENTS WITHOUT ORDERS"
)


# COMMAND ----------

# 18. IDENTIFY AND QUARANTINE SHIPMENTS WITHOUT ORDERS

silver_dfs = load_silver_tables()

shipments = silver_dfs["shipments"]
orders = silver_dfs["orders"]

shipments_without_orders = find_orphans(
    shipments,
    orders,
    "order_id",
    "order_id"
)

print(
    "Shipments without orders:",
    shipments_without_orders.count()
)

quarantine_records(
    shipments_without_orders,
    QUARANTINE_TABLES["shipments"],
    TABLES["shipments"],
    "shipment_id",
    "SHIPMENTS WITHOUT ORDERS"
)


# COMMAND ----------

# 19. IDENTIFY AND QUARANTINE INVENTORY WITHOUT PRODUCTS

silver_dfs = load_silver_tables()

inventory = silver_dfs["inventory"]
products = silver_dfs["products"]

inventory_without_products = find_orphans(
    inventory,
    products,
    "product_id",
    "product_id"
)

print(
    "Inventory without products:",
    inventory_without_products.count()
)

quarantine_records(
    inventory_without_products,
    QUARANTINE_TABLES["inventory"],
    TABLES["inventory"],
    "inventory_snapshot_id",
    "INVENTORY WITHOUT PRODUCTS"
)


# COMMAND ----------

# 20. RELOAD ALL SILVER TABLES

silver_dfs = load_silver_tables()

customers = silver_dfs["customers"]
products = silver_dfs["products"]
orders = silver_dfs["orders"]
order_items = silver_dfs["order_items"]
payments = silver_dfs["payments"]
shipments = silver_dfs["shipments"]
inventory = silver_dfs["inventory"]


# COMMAND ----------

# 21. FINAL REFERENTIAL INTEGRITY CHECKS

remaining_checks = {

    "orders_without_customers": find_orphans(
        orders,
        customers,
        "customer_id",
        "customer_id"
    ),

    "order_items_without_orders": find_orphans(
        order_items,
        orders,
        "order_id",
        "order_id"
    ),

    "order_items_without_products": find_orphans(
        order_items,
        products,
        "product_id",
        "product_id"
    ),

    "payments_without_orders": find_orphans(
        payments,
        orders,
        "order_id",
        "order_id"
    ),

    "shipments_without_orders": find_orphans(
        shipments,
        orders,
        "order_id",
        "order_id"
    ),

    "inventory_without_products": find_orphans(
        inventory,
        products,
        "product_id",
        "product_id"
    )
}


# COMMAND ----------

# 22. FINAL ORDERS WITHOUT ITEMS CHECK

orders_without_items_final = (
    orders.alias("o")
    .join(
        order_items
        .select("order_id")
        .distinct()
        .alias("i"),
        F.col("o.order_id")
        == F.col("i.order_id"),
        "left_anti"
    )
    .filter(
        F.upper(
            F.trim(
                F.coalesce(
                    F.col("o.order_status"),
                    F.lit("")
                )
            )
        ) != "CANCELLED"
    )
)

remaining_checks[
    "orders_without_items"
] = orders_without_items_final


# COMMAND ----------

# 23. FINAL ORDER TOTAL RECONCILIATION CHECK

item_totals_final = (
    order_items
    .groupBy("order_id")
    .agg(
        F.sum("line_total").alias(
            "calculated_item_total"
        )
    )
)

order_total_mismatches_final = (
    orders.alias("o")
    .join(
        item_totals_final.alias("i"),
        F.col("o.order_id")
        == F.col("i.order_id"),
        "inner"
    )
    .filter(
        F.col("o.order_total").isNotNull()
        & F.col("i.calculated_item_total").isNotNull()
        & (
            F.abs(
                F.round(
                    F.col("o.order_total")
                    - F.col("i.calculated_item_total"),
                    2
                )
            ) > F.lit(TOLERANCE)
        )
    )
)

remaining_checks[
    "order_total_reconciliation"
] = order_total_mismatches_final


# COMMAND ----------

# 24. FINAL TARGETED BUSINESS RULE REPORT

final_results = []

for rule_name, issue_df in remaining_checks.items():

    failed_count = issue_df.count()

    status = (
        "PASS"
        if failed_count == 0
        else "FAIL"
    )

    final_results.append(
        (
            rule_name,
            int(failed_count),
            status
        )
    )

report_schema = T.StructType([

    T.StructField(
        "rule_name",
        T.StringType(),
        False
    ),

    T.StructField(
        "failed_records",
        T.LongType(),
        False
    ),

    T.StructField(
        "status",
        T.StringType(),
        False
    )
])

final_report = spark.createDataFrame(
    final_results,
    schema=report_schema
)

final_report = (
    final_report
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
    .withColumn(
        "run_timestamp",
        F.lit(RUN_TIMESTAMP)
    )
    .withColumn(
        "validation_category",
        F.lit("business_rules")
    )
)

display(
    final_report.orderBy("rule_name")
)


# COMMAND ----------

# 25. SAVE FRESH REMEDIATION REPORT

(
    final_report
    .select(
        "run_id",
        "run_timestamp",
        "validation_category",
        "rule_name",
        "failed_records",
        "status"
    )
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(REMEDIATION_REPORT_TABLE)
)

print(
    "Fresh remediation report saved:",
    REMEDIATION_REPORT_TABLE
)


# COMMAND ----------

# 26. SILVER COUNT RECONCILIATION

print("=" * 80)
print("SILVER COUNT RECONCILIATION")
print("=" * 80)

after_counts = {}

for dataset, table_name in TABLES.items():

    after_count = (
        spark.table(table_name).count()
    )

    after_counts[dataset] = after_count

    removed = (
        before_counts[dataset] - after_count
    )

    print(
        f"{dataset:15} "
        f"Before={before_counts[dataset]:8,} "
        f"After={after_count:8,} "
        f"Removed={removed:8,}"
    )


# COMMAND ----------

# 27. FINAL SUMMARY

total_rules = len(final_results)

passed_rules = sum(
    1
    for _, _, status in final_results
    if status == "PASS"
)

failed_rules = (
    total_rules - passed_rules
)

total_remaining_violations = sum(
    count
    for _, count, _ in final_results
)

print("=" * 80)
print("BUSINESS RULE REMEDIATION SUMMARY")
print("=" * 80)

print(f"Run ID                      : {RUN_ID}")
print(f"Run timestamp               : {RUN_TIMESTAMP}")
print(f"Total targeted rules        : {total_rules}")
print(f"Passed rules                : {passed_rules}")
print(f"Failed rules                : {failed_rules}")
print(
    f"Remaining violating records : "
    f"{total_remaining_violations}"
)

print("-" * 80)

for rule_name, failed_count, status in final_results:

    print(
        f"{rule_name:40} "
        f"Failed={failed_count:8,} "
        f"Status={status}"
    )

print("-" * 80)

print(
    "Remediation report table:",
    REMEDIATION_REPORT_TABLE
)

if failed_rules == 0:

    print(
        "ALL 8 TARGETED BUSINESS RULES PASSED"
    )

else:

    print(
        "REMEDIATION INCOMPLETE - "
        "INVESTIGATE THE REMAINING FAILURES"
    )

print("=" * 80)


# COMMAND ----------

# 28. VERIFY THE SAVED REPORT

display(
    spark.table(REMEDIATION_REPORT_TABLE)
    .filter(
        F.col("run_id") == RUN_ID
    )
    .orderBy("rule_name")
)


# 29. OPTIONAL: INSPECT REMAINING FAILED BUSINESS KEYS

# If any rule still fails, inspect the actual offending records.
# This helps identify new issues instead of hiding them.

for rule_name, issue_df in remaining_checks.items():

    failed_count = issue_df.count()

    if failed_count > 0:

        print(
            f"Remaining records for {rule_name}: "
            f"{failed_count}"
        )

        display(
            issue_df.limit(100)
        )


# 30. COMPLETION

print("=" * 80)
print("BUSINESS RULE REMEDIATION NOTEBOOK COMPLETED")
print("=" * 80)

print(
    "Next step: rerun the original Business Rule Validation "
    "notebook to refresh its SOURCE_REPORT output."
)

print("=" * 80)



| Rule | What it checks |
|---|---|
| order_total_reconciliation | Verifies order total against the sum of order-item totals. |
| orders_without_items | Identifies orders that have no associated order items. |
| successful_payment_exceeds_order_total | Identifies successful payment amounts exceeding the order total. |
| multiple_successful_payment` | Informational check for multiple successful payments against the same order. |
| shipment_without_order | Identifies shipments referencing non-existent orders. |
| shipment_date_consistency | Validates shipment and delivery date logic. |
| selling_price_below_product_cost | Identifies products sold below their purchase cost, indicating potential losses. |
| invalid_inventory_quantity | Identifies negative or missing inventory quantities. |
| duplicate_inventory_business_key | Identifies duplicate inventory snapshots using the composite business key. |
| order_without_customer | Identifies orders referencing non-existent customers. |
| order_item_without_product | Identifies order items referencing non-existent products. |
| order_item_without_order  | Identifies order items referencing non-existent orders. |

In [0]:
%sql
SELECT
    run_id,
    rule_name,
    failed_records,
    status
FROM workspace.ecommerce_silver.business_rule_remediation_results
WHERE run_id = (
    SELECT MAX(run_id)
    FROM workspace.ecommerce_silver.business_rule_remediation_results
)
ORDER BY rule_name;

In [0]:
# Display all failed business rules

failed_rules_df = (
    spark.table(
        "workspace.ecommerce_silver.business_rule_validation_report"
    )
    .filter(F.col("status") == "FAIL")
    .orderBy("rule_name")
)

display(failed_rules_df)

In [0]:
# Display the number of records failing each rule

failed_rules_df.select(
    "rule_name",
    "dataset_name",
    "failed_records",
    "failure_percentage",
    "description"
).show(50, truncate=False)

In [0]:
# ============================================================
# BUSINESS RULE FAILURE DIAGNOSTICS
# ============================================================

# 1. Order items referencing missing orders
orphan_order_items = (
    order_items_df
    .join(
        orders_df.select("order_id").distinct(),
        "order_id",
        "left_anti"
    )
)

print("1. ORDER ITEMS WITHOUT ORDERS")
display(
    orphan_order_items.select(
        "order_item_id", "order_id", "product_id"
    )
)

# 2. Order items referencing missing products
orphan_products = (
    order_items_df
    .join(
        products_df.select("product_id").distinct(),
        "product_id",
        "left_anti"
    )
)

print("2. ORDER ITEMS WITHOUT PRODUCTS")
display(
    orphan_products.select(
        "order_item_id", "order_id", "product_id"
    )
)

# 3. Orders referencing missing customers
orphan_customers = (
    orders_df
    .join(
        customers_df.select("customer_id").distinct(),
        "customer_id",
        "left_anti"
    )
)

print("3. ORDERS WITHOUT CUSTOMERS")
display(
    orphan_customers.select(
        "order_id", "customer_id", "order_status"
    )
)

# 4. Check whether missing customers exist in quarantine
quarantine_df = spark.table(
    "workspace.ecommerce_silver.quarantine_customers"
)

orders_with_quarantined_customers = (
    orphan_customers
    .join(
        quarantine_df.select("customer_id").distinct(),
        "customer_id",
        "inner"
    )
)

print("4. ORDERS LINKED TO QUARANTINED CUSTOMERS")
display(
    orders_with_quarantined_customers.select(
        "order_id", "customer_id"
    )
)

# 5. Orders without order items
orders_without_items = (
    orders_df
    .join(
        order_items_df.select("order_id").distinct(),
        "order_id",
        "left_anti"
    )
)

print("5. ORDERS WITHOUT ORDER ITEMS")
display(
    orders_without_items.select(
        "order_id", "customer_id", "order_total"
    )
)

# 6. Order total mismatches
item_totals = (
    order_items_df
    .groupBy("order_id")
    .agg(
        F.sum("line_total").alias("calculated_order_total")
    )
)

order_mismatches = (
    orders_df
    .join(item_totals, "order_id", "inner")
    .filter(
        F.abs(
            F.col("order_total") -
            F.col("calculated_order_total")
        ) > F.lit(0.02)
    )
)

print("6. ORDER TOTAL MISMATCHES")
display(
    order_mismatches.select(
        "order_id",
        "order_total",
        "calculated_order_total"
    )
)

# 7. Shipments referencing missing orders
orphan_shipments = (
    shipments_df
    .join(
        orders_df.select("order_id").distinct(),
        "order_id",
        "left_anti"
    )
)

print("7. SHIPMENTS WITHOUT ORDERS")
display(
    orphan_shipments.select(
        "shipment_id", "order_id", "shipment_status"
    )
)

print("DIAGNOSTIC CHECKS COMPLETED")

In [0]:
%sql
SELECT
    rule_name,
    dataset_name,
    total_records,
    failed_records,
    failure_percentage,
    status,
    severity,
    description
FROM workspace.ecommerce_silver.business_rule_validation_report
WHERE status IN ('FAIL', 'WARN')
ORDER BY
    CASE status
        WHEN 'FAIL' THEN 1
        WHEN 'WARN' THEN 2
    END,
    dataset_name,
    rule_name;

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_silver.business_rule_validation_report
ORDER BY dataset_name, rule_name;


**A. Find order items referencing missing orders**

In [0]:
%sql

SELECT oi.*
FROM workspace.ecommerce_silver.silver_order_items oi
LEFT ANTI JOIN workspace.ecommerce_silver.silver_orders o
    ON oi.order_id = o.order_id;


**B. Find order items referencing missing products**

In [0]:
%sql

SELECT oi.*
FROM workspace.ecommerce_silver.silver_order_items oi
LEFT ANTI JOIN workspace.ecommerce_silver.silver_products p
    ON oi.product_id = p.product_id;


**C. Find inventory snapshots referencing missing products**

In [0]:
%sql

SELECT i.*
FROM workspace.ecommerce_silver.silver_inventory i
LEFT ANTI JOIN workspace.ecommerce_silver.silver_products p
    ON i.product_id = p.product_id;


**D. Find orders referencing missing customers**

In [0]:
%sql

SELECT o.*
FROM workspace.ecommerce_silver.silver_orders o
LEFT ANTI JOIN workspace.ecommerce_silver.silver_customers c
    ON o.customer_id = c.customer_id;


**E. Find the order without valid items**

In [0]:
%sql

SELECT
    o.order_id,
    o.order_status,
    o.order_total
FROM workspace.ecommerce_silver.silver_orders o
LEFT ANTI JOIN workspace.ecommerce_silver.silver_order_items oi
    ON o.order_id = oi.order_id
WHERE LOWER(TRIM(o.order_status)) <> 'cancelled';


**F. Find payments and shipments referencing missing orders**

In [0]:
%sql
SELECT p.*
FROM workspace.ecommerce_silver.silver_payments p
LEFT ANTI JOIN workspace.ecommerce_silver.silver_orders o
    ON p.order_id = o.order_id;


**Investigate the missing products first**

In [0]:
%sql
SELECT
    b.product_id,
    COUNT(*) AS bronze_record_count
FROM workspace.ecommerce_bronze.bronze_products b
LEFT ANTI JOIN workspace.ecommerce_silver.silver_products s
    ON b.product_id = s.product_id
GROUP BY b.product_id
ORDER BY b.product_id;


**Identify the single order without items**

In [0]:
%sql
SELECT
    o.order_id,
    o.order_status,
    o.order_total
FROM workspace.ecommerce_silver.silver_orders o
LEFT ANTI JOIN workspace.ecommerce_silver.silver_order_items oi
    ON o.order_id = oi.order_id
WHERE UPPER(TRIM(o.order_status)) <> 'CANCELLED';


**Two more outputs are still needed**

In [0]:
%sql
SELECT p.*
FROM workspace.ecommerce_silver.silver_payments p
LEFT ANTI JOIN workspace.ecommerce_silver.silver_orders o
    ON p.order_id = o.order_id;


**Run this SQL to investigate missing order references**

In [0]:
%sql
SELECT
    p.payment_id,
    p.order_id,
    p.payment_method,
    p.payment_status,
    p.amount
FROM workspace.ecommerce_silver.silver_payments p
LEFT ANTI JOIN workspace.ecommerce_silver.silver_orders o
    ON p.order_id = o.order_id
ORDER BY p.order_id;


**Check the business-rule report again**

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_silver.business_rule_validation_report;

In [0]:
# --------------------------------------------------
# CUSTOMER SILVER RECONCILIATION
# --------------------------------------------------

print("1. Missing customer IDs")
display(missing_silver_customers)

print("2. Bronze matching records")
display(
    bronze_customers.join(
        missing_silver_customers,
        on="customer_id",
        how="inner"
    )
)

print("3. Silver customer table count")
print(silver_customers.count())

print("4. Bronze customer table count")
print(bronze_customers.count())

print("5. Customer quarantine count")
print(customer_quarantine.count())

print("6. Customer quarantine reasons")
display(
    customer_quarantine
    .groupBy("quarantine_reason")
    .count()
)

In [0]:
bronze_customer_ids = bronze_customers.select(
    "customer_id"
).distinct()

silver_customer_ids = silver_customers.select(
    "customer_id"
).distinct()

missing_from_silver = bronze_customer_ids.join(
    silver_customer_ids,
    on="customer_id",
    how="left_anti"
)

print("Bronze customer IDs missing from Silver:")
print(missing_from_silver.count())

display(missing_from_silver)

In [0]:
# --------------------------------------------------
# CUSTOMER COUNT RECONCILIATION
# --------------------------------------------------

print("BRONZE TOTAL ROWS:", bronze_customers.count())

print(
    "BRONZE DISTINCT CUSTOMER IDS:",
    bronze_customers.select("customer_id").distinct().count()
)

print("SILVER TOTAL ROWS:", silver_customers.count())

print(
    "SILVER DISTINCT CUSTOMER IDS:",
    silver_customers.select("customer_id").distinct().count()
)

print("BRONZE DUPLICATE CUSTOMER IDS:")

bronze_duplicates = (
    bronze_customers
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

display(
    bronze_duplicates.orderBy(F.col("count").desc())
)

print("ALL BRONZE CUSTOMERS MISSING FROM SILVER:")

display(missing_from_silver)

In [0]:
print("COMPLETE MISSING CUSTOMER RECORDS:")

display(
    bronze_customers.join(
        missing_from_silver,
        on="customer_id",
        how="inner"
    )
)

In [0]:
# COMMAND ----------

orders = spark.table(
    "workspace.ecommerce_silver.silver_orders"
)

print("SILVER ORDERS COLUMNS")
print(orders.columns)

print("\nSILVER ORDERS SCHEMA")
orders.printSchema()

print("\nPOSSIBLE STATUS COLUMNS")

status_columns = [
    column
    for column in orders.columns
    if "status" in column.lower()
]

print(status_columns)

print("\nSAMPLE ORDER RECORDS")

display(orders.limit(10))

In [0]:
# COMMAND ----------

table_names = [
    "silver_customers",
    "silver_products",
    "silver_orders",
    "silver_order_items",
    "silver_payments",
    "silver_shipments",
    "silver_inventory"
]

for table_name in table_names:

    full_table_name = (
        f"workspace.ecommerce_silver.{table_name}"
    )

    df = spark.table(full_table_name)

    print("\n" + "=" * 65)
    print(table_name.upper())
    print("=" * 65)

    print(df.columns)